# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Cooper30/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
### Lane confirmation

I am keeping my lane: **Refresh / Content Opportunity Scoring**.

The decision I want to support is: which content items should an editor review first for a possible refresh?

Before writing the baseline rule, I will check the two signals the rule depends on:

1. **Staleness** — older content may be more likely to need a refresh.
2. **Visibility / volume** — a stale page is more important to review if it is still receiving meaningful search impressions.

The candidate rule is intentionally simple:

> Review content that is at least 180 days old and still receives at least 500 impressions in the March 2026 observation window.

I will only keep this rule if the signal checks below give reasonable support for it.

In [1]:
%pip -q install duckdb huggingface_hub

import duckdb
import pandas as pd
import numpy as np

from google.colab import userdata


# ==================================================
# SETUP
# ==================================================

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN bulunamadı. Colab Secrets bölümüne "
        "HF_TOKEN adıyla Hugging Face READ token ekle."
    )

con = duckdb.connect()

safe_token = HF_TOKEN.replace("'", "''")

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{safe_token}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

FACT_MARCH = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    f")"
)

DIM_CONTENT = (
    f"read_parquet("
    f"'{REL}/dim_content.parquet'"
    f")"
)

print("Warehouse connection ready.")
print("Observation window: March 2026")


# ==================================================
# BUILD PAGE-LEVEL FRAME
# one row = client x content
# ==================================================

page_march = con.execute(f"""
WITH march AS (

    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS impressions_march,

        SUM(gsc_clicks) AS clicks_march,

        SUM(gsc_clicks) * 1.0
            / NULLIF(SUM(gsc_impressions), 0)
            AS ctr_march,

        SUM(gsc_sum_position) * 1.0
            / NULLIF(SUM(gsc_impressions), 0)
            AS avg_position_march,

        COUNT(DISTINCT report_date)
            AS observed_days_march

    FROM {FACT_MARCH}

    WHERE gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    m.*,

    DATEDIFF(
        'day',
        c.content_created_date,
        DATE '2026-03-31'
    ) AS content_age_days

FROM march m

LEFT JOIN {DIM_CONTENT} c
    ON m.content_hash_id = c.content_hash_id

WHERE c.content_created_date IS NOT NULL
  AND c.content_created_date <= DATE '2026-03-31'
""").df()


print(f"\nPage-level rows: {len(page_march):,}")

display(page_march.head())


# ==================================================
# SIGNAL 1 — STALENESS
# ==================================================

page_march["staleness_bucket"] = pd.cut(
    page_march["content_age_days"],
    bins=[
        -np.inf,
        89,
        179,
        np.inf
    ],
    labels=[
        "fresh_<90d",
        "moderate_90_179d",
        "stale_180d_plus"
    ]
)


staleness_check = (
    page_march
    .groupby(
        "staleness_bucket",
        observed=True
    )
    .agg(

        n=(
            "content_hash_id",
            "size"
        ),

        median_impressions=(
            "impressions_march",
            "median"
        ),

        mean_impressions=(
            "impressions_march",
            "mean"
        ),

        visible_500_plus=(
            "impressions_march",
            lambda s: int(
                (s >= 500).sum()
            )
        ),

        visible_rate_500_plus=(
            "impressions_march",
            lambda s: float(
                (s >= 500).mean()
            )
        )

    )
    .reset_index()
)


print("\n==============================")
print("SIGNAL 1 — STALENESS")
print("==============================")

display(staleness_check)

print(
    "n =",
    f"{staleness_check['n'].sum():,}"
)


# ==================================================
# SIGNAL 2 — VISIBILITY / VOLUME
# ==================================================

page_march["volume_bucket"] = pd.cut(
    page_march["impressions_march"],
    bins=[
        -0.1,
        99,
        499,
        4999,
        np.inf
    ],
    labels=[
        "<100",
        "100-499",
        "500-4,999",
        "5,000+"
    ]
)


volume_check = (
    page_march
    .groupby(
        "volume_bucket",
        observed=True
    )
    .agg(

        n=(
            "content_hash_id",
            "size"
        ),

        median_clicks=(
            "clicks_march",
            "median"
        ),

        median_ctr=(
            "ctr_march",
            "median"
        ),

        median_position=(
            "avg_position_march",
            "median"
        ),

        median_observed_days=(
            "observed_days_march",
            "median"
        )

    )
    .reset_index()
)


print("\n==============================")
print("SIGNAL 2 — VISIBILITY / VOLUME")
print("==============================")

display(volume_check)

print(
    "n =",
    f"{volume_check['n'].sum():,}"
)

Warehouse connection ready.
Observation window: March 2026


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Page-level rows: 176,738


,client_hash_id,content_hash_id,impressions_march,clicks_march,ctr_march,avg_position_march,observed_days_march,content_age_days
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,0.001073,6.893301,31,396
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,0.000000,3.214128,31,396
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,0.001066,6.535346,31,396
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,0.002629,7.435680,31,396
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,429.0,1.0,0.002331,3.871795,31,396



SIGNAL 1 — STALENESS


,staleness_bucket,n,median_impressions,mean_impressions,visible_500_plus,visible_rate_500_plus
0,fresh_<90d,57735,175.0,1387.538183,20012,0.346618
1,moderate_90_179d,25872,251.0,2194.892664,10448,0.403834
2,stale_180d_plus,93131,155.0,1543.651512,31464,0.337847


n = 176,738

SIGNAL 2 — VISIBILITY / VOLUME


,volume_bucket,n,median_clicks,median_ctr,median_position,median_observed_days
0,<100,75297,0.0,0.000000,8.142857,7.0
1,100-499,39517,0.0,0.000000,11.556522,29.0
2,"500-4,999",48632,2.0,0.001715,7.111987,31.0
3,"5,000+",13292,20.0,0.002041,5.529753,31.0


n = 176,738


### Signal verdicts

**Staleness — MIXED**

The relationship is not monotonic. The 90–179 day bucket has the highest median impressions (251) and the highest share of pages with at least 500 impressions (40.38%). Content aged 180+ days has a lower median of 155 impressions and a 33.78% visible rate.

This means I should not assume that older content automatically represents a stronger opportunity. I will therefore use staleness as a simple eligibility condition rather than giving it heavy ranking weight.

**Volume — CONFIRMED**

The volume buckets show a clear difference in actual search activity. Pages below 500 impressions have a median of 0 clicks, while pages with 500–4,999 impressions have a median of 2 clicks and pages with 5,000+ impressions have a median of 20 clicks.

The higher-volume buckets also have a median of 31 observed days, so the signal is not mainly caused by incomplete observation windows.

Based on these checks, the baseline will prioritize content that is both stale and still visibly active in search.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
from pathlib import Path


# ==================================================
# BASELINE RULE
# ==================================================
#
# Core rule:
# - content age >= 180 days
# - March impressions >= 500
#
# Score:
# +2 = stale (>=180 days)
# +2 = visible (>=500 impressions)
# +1 = very stale (>=365 days)
# +1 = high volume (>=5,000 impressions)
#
# Therefore:
# score >= 4 means both core conditions are satisfied.
#
# No label-derived or future-window inputs are used.
# ==================================================


baseline = page_march.copy()

baseline["impressions_march"] = (
    baseline["impressions_march"]
    .fillna(0)
)

baseline["content_age_days"] = (
    baseline["content_age_days"]
    .fillna(0)
)


# --------------------------------------------------
# Hand-written score
# --------------------------------------------------

baseline["baseline_action_score"] = 0


# Core staleness condition
baseline.loc[
    baseline["content_age_days"] >= 180,
    "baseline_action_score"
] += 2


# Core visibility condition
baseline.loc[
    baseline["impressions_march"] >= 500,
    "baseline_action_score"
] += 2


# Small bonus: very stale
baseline.loc[
    baseline["content_age_days"] >= 365,
    "baseline_action_score"
] += 1


# Small bonus: high search volume
baseline.loc[
    baseline["impressions_march"] >= 5000,
    "baseline_action_score"
] += 1


# --------------------------------------------------
# ONE reason code per row
# --------------------------------------------------

baseline["reason_code"] = np.where(
    baseline["baseline_action_score"] >= 4,
    "STALE_VISIBLE",
    "BELOW_CORE_RULE"
)


# --------------------------------------------------
# Action label
# --------------------------------------------------

baseline["action_label"] = np.where(
    baseline["baseline_action_score"] >= 4,
    "REVIEW_FOR_REFRESH",
    "DEFER"
)


# --------------------------------------------------
# Rank the complete queue
# --------------------------------------------------

baseline = (
    baseline
    .sort_values(
        by=[
            "baseline_action_score",
            "impressions_march",
            "content_age_days"
        ],
        ascending=[
            False,
            False,
            False
        ]
    )
    .reset_index(drop=True)
)

baseline["baseline_rank"] = (
    np.arange(1, len(baseline) + 1)
)


# --------------------------------------------------
# Final queue columns
# --------------------------------------------------

queue = baseline[
    [
        "baseline_rank",
        "client_hash_id",
        "content_hash_id",
        "baseline_action_score",
        "reason_code",
        "action_label",
        "content_age_days",
        "impressions_march",
        "clicks_march",
        "ctr_march",
        "avg_position_march",
        "observed_days_march"
    ]
].copy()


# --------------------------------------------------
# Write required CSV
# --------------------------------------------------

OUTPUT_PATH = Path(
    "work/outputs/baseline_action_score.csv"
)

OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

queue.to_csv(
    OUTPUT_PATH,
    index=False
)


# --------------------------------------------------
# Checks
# --------------------------------------------------

print(
    f"Rows ranked: {len(queue):,}"
)

print(
    "Review for refresh:",
    f"{(queue['action_label'] == 'REVIEW_FOR_REFRESH').sum():,}"
)

print(
    "Deferred:",
    f"{(queue['action_label'] == 'DEFER').sum():,}"
)

print(
    f"\nCSV written to: {OUTPUT_PATH}"
)


# --------------------------------------------------
# Show top 10
# --------------------------------------------------

print("\nTOP 10 BASELINE QUEUE")

display(
    queue.head(10)
)

Rows ranked: 176,738
Review for refresh: 31,464
Deferred: 145,274

CSV written to: work/outputs/baseline_action_score.csv

TOP 10 BASELINE QUEUE


,baseline_rank,client_hash_id,content_hash_id,baseline_action_score,reason_code,action_label,content_age_days,impressions_march,clicks_march,ctr_march,avg_position_march,observed_days_march
0,1,client_e547b89c05043229,content_eadb33b5df496f4a,6,STALE_VISIBLE,REVIEW_FOR_REFRESH,375,617124.0,5668.0,0.009185,2.331470,29
1,2,client_e547b89c05043229,content_ec2e0346994fb5a5,6,STALE_VISIBLE,REVIEW_FOR_REFRESH,434,245276.0,1480.0,0.006034,2.757730,29
2,3,client_e547b89c05043229,content_0e03de7680314cd5,6,STALE_VISIBLE,REVIEW_FOR_REFRESH,375,221310.0,720.0,0.003253,2.506100,29
3,4,client_e547b89c05043229,content_8d7d99f109e19aa2,6,STALE_VISIBLE,REVIEW_FOR_REFRESH,375,203497.0,289.0,0.001420,2.468557,29
4,5,client_e547b89c05043229,content_4ffe18112a5642e3,6,STALE_VISIBLE,REVIEW_FOR_REFRESH,375,186983.0,586.0,0.003134,2.389966,29
5,6,client_73cda7b4e4f265ea,content_471d9cabce329a66,6,STALE_VISIBLE,REVIEW_FOR_REFRESH,375,164885.0,396.0,0.002402,4.603724,31
6,7,client_73cda7b4e4f265ea,content_fd2117c2c6790e4b,6,STALE_VISIBLE,REVIEW_FOR_REFRESH,410,151166.0,408.0,0.002699,3.428906,31
7,8,client_73cda7b4e4f265ea,content_e241d6415ac9e534,6,STALE_VISIBLE,REVIEW_FOR_REFRESH,412,142304.0,343.0,0.002410,3.286851,31
8,9,client_73cda7b4e4f265ea,content_8e1334d6356668e3,6,STALE_VISIBLE,REVIEW_FOR_REFRESH,410,134984.0,1.0,0.000007,2.693038,31
9,10,client_73cda7b4e4f265ea,content_00d4fdf6e48a2d38,6,STALE_VISIBLE,REVIEW_FOR_REFRESH,410,126836.0,467.0,0.003682,5.310314,31


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [3]:
# ==================================================
# TOP-10 MANUAL REVIEW
# ==================================================

top10_review = queue.head(10).copy()


def explain_why(row):
    reasons = []

    if row["content_age_days"] >= 365:
        reasons.append(
            f"very stale ({int(row['content_age_days'])} days old)"
        )
    elif row["content_age_days"] >= 180:
        reasons.append(
            f"stale ({int(row['content_age_days'])} days old)"
        )

    if row["impressions_march"] >= 5000:
        reasons.append(
            f"high search volume ({int(row['impressions_march']):,} March impressions)"
        )
    elif row["impressions_march"] >= 500:
        reasons.append(
            f"still visible ({int(row['impressions_march']):,} March impressions)"
        )

    return " + ".join(reasons)


def what_would_make_it_wrong(row):
    return (
        "Wrong if the page is intentionally evergreen/current, "
        "the created date does not reflect its latest update, "
        "or the search volume does not represent a real refresh opportunity."
    )


top10_review["why_its_here"] = top10_review.apply(
    explain_why,
    axis=1
)

top10_review["what_would_make_it_wrong"] = top10_review.apply(
    what_would_make_it_wrong,
    axis=1
)


review_table = top10_review[
    [
        "baseline_rank",
        "content_hash_id",
        "action_label",
        "why_its_here",
        "what_would_make_it_wrong"
    ]
]


display(review_table)


print("\nTOP-10 REVIEW — ONE LINE EACH\n")

for _, row in top10_review.iterrows():

    print(
        f"#{int(row['baseline_rank'])} | "
        f"{row['content_hash_id']} | "
        f"Action: {row['action_label']} | "
        f"Why: {row['why_its_here']} | "
        f"What would make it wrong: "
        f"{row['what_would_make_it_wrong']}"
    )

,baseline_rank,content_hash_id,action_label,why_its_here,what_would_make_it_wrong
0,1,content_eadb33b5df496f4a,REVIEW_FOR_REFRESH,very stale (375 days old) + high search volume...,Wrong if the page is intentionally evergreen/c...
1,2,content_ec2e0346994fb5a5,REVIEW_FOR_REFRESH,very stale (434 days old) + high search volume...,Wrong if the page is intentionally evergreen/c...
2,3,content_0e03de7680314cd5,REVIEW_FOR_REFRESH,very stale (375 days old) + high search volume...,Wrong if the page is intentionally evergreen/c...
3,4,content_8d7d99f109e19aa2,REVIEW_FOR_REFRESH,very stale (375 days old) + high search volume...,Wrong if the page is intentionally evergreen/c...
4,5,content_4ffe18112a5642e3,REVIEW_FOR_REFRESH,very stale (375 days old) + high search volume...,Wrong if the page is intentionally evergreen/c...
5,6,content_471d9cabce329a66,REVIEW_FOR_REFRESH,very stale (375 days old) + high search volume...,Wrong if the page is intentionally evergreen/c...
6,7,content_fd2117c2c6790e4b,REVIEW_FOR_REFRESH,very stale (410 days old) + high search volume...,Wrong if the page is intentionally evergreen/c...
7,8,content_e241d6415ac9e534,REVIEW_FOR_REFRESH,very stale (412 days old) + high search volume...,Wrong if the page is intentionally evergreen/c...
8,9,content_8e1334d6356668e3,REVIEW_FOR_REFRESH,very stale (410 days old) + high search volume...,Wrong if the page is intentionally evergreen/c...
9,10,content_00d4fdf6e48a2d38,REVIEW_FOR_REFRESH,very stale (410 days old) + high search volume...,Wrong if the page is intentionally evergreen/c...



TOP-10 REVIEW — ONE LINE EACH

#1 | content_eadb33b5df496f4a | Action: REVIEW_FOR_REFRESH | Why: very stale (375 days old) + high search volume (617,124 March impressions) | What would make it wrong: Wrong if the page is intentionally evergreen/current, the created date does not reflect its latest update, or the search volume does not represent a real refresh opportunity.
#2 | content_ec2e0346994fb5a5 | Action: REVIEW_FOR_REFRESH | Why: very stale (434 days old) + high search volume (245,276 March impressions) | What would make it wrong: Wrong if the page is intentionally evergreen/current, the created date does not reflect its latest update, or the search volume does not represent a real refresh opportunity.
#3 | content_0e03de7680314cd5 | Action: REVIEW_FOR_REFRESH | Why: very stale (375 days old) + high search volume (221,310 March impressions) | What would make it wrong: Wrong if the page is intentionally evergreen/current, the created date does not reflect its latest update, or t

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [4]:
# ==================================================
# 4. WEAK PICKS
# ==================================================
# These are the selected pages closest to the rule boundary.
# They satisfy the core rule, but receive no bonus for
# being very stale or very high volume.
# ==================================================

weak_picks = (
    queue[
        (queue["action_label"] == "REVIEW_FOR_REFRESH")
        & (queue["baseline_action_score"] == 4)
    ]
    .sort_values(
        by=[
            "impressions_march",
            "content_age_days"
        ],
        ascending=[
            True,
            True
        ]
    )
    .head(10)
    .copy()
)


weak_picks["why_weak"] = weak_picks.apply(
    lambda row:
        (
            f"Only just clears the baseline rule: "
            f"{int(row['content_age_days'])} days old and "
            f"{int(row['impressions_march']):,} March impressions."
        ),
    axis=1
)


weak_picks["what_to_check"] = (
    "Check whether the page actually needs updating, "
    "whether its age reflects the last meaningful update, "
    "and whether the search demand is stable enough to justify editor time."
)


display(
    weak_picks[
        [
            "baseline_rank",
            "content_hash_id",
            "baseline_action_score",
            "content_age_days",
            "impressions_march",
            "why_weak",
            "what_to_check"
        ]
    ]
)


print("\nWEAK PICKS — BOUNDARY CASES\n")

for _, row in weak_picks.iterrows():

    print(
        f"#{int(row['baseline_rank'])} | "
        f"{row['content_hash_id']} | "
        f"{row['why_weak']} | "
        f"What to check: {row['what_to_check']}"
    )

,baseline_rank,content_hash_id,baseline_action_score,content_age_days,impressions_march,why_weak,what_to_check
31461,31462,content_47d195532381032e,4,193,500.0,Only just clears the baseline rule: 193 days o...,Check whether the page actually needs updating...
31462,31463,content_8779cebd5ce32164,4,193,500.0,Only just clears the baseline rule: 193 days o...,Check whether the page actually needs updating...
31463,31464,content_47aac4e48b80e14b,4,193,500.0,Only just clears the baseline rule: 193 days o...,Check whether the page actually needs updating...
31460,31461,content_024e8a995be113ee,4,205,500.0,Only just clears the baseline rule: 205 days o...,Check whether the page actually needs updating...
31458,31459,content_ef61c1306fc093b0,4,210,500.0,Only just clears the baseline rule: 210 days o...,Check whether the page actually needs updating...
31459,31460,content_e96ed438e5e71cb8,4,210,500.0,Only just clears the baseline rule: 210 days o...,Check whether the page actually needs updating...
31455,31456,content_edc783a3b3083631,4,214,500.0,Only just clears the baseline rule: 214 days o...,Check whether the page actually needs updating...
31456,31457,content_e7282049a51541af,4,214,500.0,Only just clears the baseline rule: 214 days o...,Check whether the page actually needs updating...
31457,31458,content_b361a2253d66e155,4,214,500.0,Only just clears the baseline rule: 214 days o...,Check whether the page actually needs updating...
31454,31455,content_855b5b2360cec31a,4,230,500.0,Only just clears the baseline rule: 230 days o...,Check whether the page actually needs updating...



WEAK PICKS — BOUNDARY CASES

#31462 | content_47d195532381032e | Only just clears the baseline rule: 193 days old and 500 March impressions. | What to check: Check whether the page actually needs updating, whether its age reflects the last meaningful update, and whether the search demand is stable enough to justify editor time.
#31463 | content_8779cebd5ce32164 | Only just clears the baseline rule: 193 days old and 500 March impressions. | What to check: Check whether the page actually needs updating, whether its age reflects the last meaningful update, and whether the search demand is stable enough to justify editor time.
#31464 | content_47aac4e48b80e14b | Only just clears the baseline rule: 193 days old and 500 March impressions. | What to check: Check whether the page actually needs updating, whether its age reflects the last meaningful update, and whether the search demand is stable enough to justify editor time.
#31461 | content_024e8a995be113ee | Only just clears the baseline r

### Self-check

- [x] Two signals checked with visible bucket tables and `n`
- [x] At least one signal is linked to a real FlyRank flag
- [x] Staleness verdict: **MIXED**
- [x] Volume verdict: **CONFIRMED**
- [x] One hand-written baseline rule
- [x] One score per row
- [x] One reason code per row
- [x] One action label per row
- [x] Full ranked queue generated
- [x] `work/outputs/baseline_action_score.csv` written from the notebook
- [x] Top 10 manually reviewed
- [x] Weak / boundary picks inspected
- [x] No future-window inputs
- [x] No label-derived inputs
- [x] Run receipt written to `work/outputs/w04_baseline_metrics.json`

**Lane confirmation:** Refresh / Content Opportunity Scoring.

In [5]:
# ==================================================
# 5. SELF-CHECK + RUN RECEIPT
# ==================================================

import json
from pathlib import Path


review_count = int(
    (queue["action_label"] == "REVIEW_FOR_REFRESH").sum()
)

defer_count = int(
    (queue["action_label"] == "DEFER").sum()
)

total_rows = int(len(queue))


# --------------------------------------------------
# Required checks
# --------------------------------------------------

assert total_rows == len(page_march)

assert queue["baseline_rank"].is_unique

assert queue["baseline_rank"].min() == 1

assert queue["baseline_rank"].max() == total_rows

assert queue["reason_code"].notna().all()

assert queue["action_label"].notna().all()


# Every selected row must satisfy BOTH core conditions.
selected = queue[
    queue["action_label"] == "REVIEW_FOR_REFRESH"
]

assert (
    selected["content_age_days"] >= 180
).all()

assert (
    selected["impressions_march"] >= 500
).all()


# No future-window or label-derived inputs.
forbidden_inputs = {
    "label",
    "target",
    "trend_pct",
    "trend_direction",
    "future_clicks",
    "future_impressions"
}

used_inputs = {
    "content_age_days",
    "impressions_march"
}

assert forbidden_inputs.isdisjoint(used_inputs)


# --------------------------------------------------
# Run receipt
# --------------------------------------------------

metrics = {
    "lane": "Refresh / Content Opportunity Scoring",
    "observation_window": "2026-03",

    "total_rows_ranked": total_rows,
    "review_for_refresh": review_count,
    "deferred": defer_count,

    "signal_verdicts": {
        "staleness": "MIXED",
        "volume": "CONFIRMED"
    },

    "rule": {
        "minimum_age_days": 180,
        "minimum_impressions": 500,
        "very_stale_bonus_days": 365,
        "high_volume_bonus_impressions": 5000
    },

    "inputs_used": [
        "content_age_days",
        "impressions_march"
    ],

    "future_window_inputs_used": False,
    "label_derived_inputs_used": False
}


METRICS_PATH = Path(
    "work/outputs/w04_baseline_metrics.json"
)

METRICS_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

with open(
    METRICS_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        metrics,
        f,
        indent=2
    )


print("SELF-CHECK PASSED ✅")
print(f"Rows ranked: {total_rows:,}")
print(f"Review for refresh: {review_count:,}")
print(f"Deferred: {defer_count:,}")
print("Future-window inputs used: NO")
print("Label-derived inputs used: NO")
print(f"Metrics written to: {METRICS_PATH}")

SELF-CHECK PASSED ✅
Rows ranked: 176,738
Review for refresh: 31,464
Deferred: 145,274
Future-window inputs used: NO
Label-derived inputs used: NO
Metrics written to: work/outputs/w04_baseline_metrics.json
